In [2]:
!pip install pandas sqlalchemy psycopg2-binary

In [3]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2
import os

In [4]:
df = pd.read_csv(r'C:\Users\Admin\Desktop\10Alytics\flour4four_etl_with_airflow\raw_dataset\flour4four_orders_oct2025.csv')

In [5]:
df.head()

,order_id,order_date,delivery_date,business_id,business_name,business_type,business_address,contact_name,contact_phone,flour_type,quantity_bags,price_per_bag,total_amount,payment_method,order_status,rider_name,rider_phone
0,ORD-214576,2025-10-25,2025-10-25,BIZ-1018,"Adams, Zuniga and Wong",Restaurant,"Herbert Macaulay Way, Abuja",Elimu Agbaje,8017507864,NaN,26,9500.0,247000,POS,Delivered,Aisha Bello,8089864260
1,ORD-299448,2025-10-08,2025-10-08,BIZ-1006,Blake and Sons,Bakery,"Ahmadu Bello Way, Abuja",Bolanle Kalumba,8055667651,Bread Flour,27,10000.0,270000,POS,Cancelled,Tunde Oladipo,8019121552
2,ORD-246991,NaN,2025-10-17,BIZ-1052,Chapman and Sons,Cafe,"Ahmadu Bello Way, Abuja",Hassan Nyoni,8083863413,Pastry Flour,21,9800.0,205800,Bank Transfer,Cancelled,Tunde Oladipo,8019121552
3,ORD-392075,2025-10-13,2025-10-13,BIZ-1035,Rodriguez-Graham,Restaurant,"Herbert Macaulay Way, Abuja",Mandela Onyango,8075228535,All-purpose,20,10500.0,210000,Bank Transfer,Pending,Tunde Oladipo,8019121552
4,ORD-179046,2025-10-14,2025-10-14,BIZ-1039,"Romero, Gonzalez and Brooks",NaN,"Garki Area 1, Abuja",Mojisola Seko,8060119651,Bread Flour,40,NaN,380000,POS,Pending,Emeka John,8019196777


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          10000 non-null  object 
 1   order_date        9372 non-null   object 
 2   delivery_date     10000 non-null  object 
 3   business_id       10000 non-null  object 
 4   business_name     9398 non-null   object 
 5   business_type     9013 non-null   object 
 6   business_address  9022 non-null   object 
 7   contact_name      10000 non-null  object 
 8   contact_phone     10000 non-null  int64  
 9   flour_type        9358 non-null   object 
 10  quantity_bags     10000 non-null  int64  
 11  price_per_bag     9364 non-null   float64
 12  total_amount      10000 non-null  int64  
 13  payment_method    10000 non-null  object 
 14  order_status      10000 non-null  object 
 15  rider_name        10000 non-null  object 
 16  rider_phone       10000 non-null  int64  

In [7]:
df.isnull().sum()

order_id              0
order_date          628
delivery_date         0
business_id           0
business_name       602
business_type       987
business_address    978
contact_name          0
contact_phone         0
flour_type          642
quantity_bags         0
price_per_bag       636
total_amount          0
payment_method        0
order_status          0
rider_name            0
rider_phone           0
dtype: int64

In [8]:

df['order_date'] = df['order_date'].fillna(df['delivery_date'])
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

In [9]:

df['business_name'] = df['business_name'].fillna('Unknown')
df['business_type'] = df['business_type'].fillna('Unknown')
df['business_address'] = df['business_address'].fillna('Unknown')
df['flour_type'] = df['flour_type'].fillna('Unknown')
df['price_per_bag'] = df['price_per_bag'].fillna(df['price_per_bag'].median())



In [10]:
df['rider_phone'] = df['rider_phone'].astype(str)
df['total_amount'] = df['total_amount'].astype(float)
df['contact_phone'] = df['contact_phone'].astype(str)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          10000 non-null  object        
 1   order_date        10000 non-null  datetime64[ns]
 2   delivery_date     10000 non-null  datetime64[ns]
 3   business_id       10000 non-null  object        
 4   business_name     10000 non-null  object        
 5   business_type     10000 non-null  object        
 6   business_address  10000 non-null  object        
 7   contact_name      10000 non-null  object        
 8   contact_phone     10000 non-null  object        
 9   flour_type        10000 non-null  object        
 10  quantity_bags     10000 non-null  int64         
 11  price_per_bag     10000 non-null  float64       
 12  total_amount      10000 non-null  float64       
 13  payment_method    10000 non-null  object        
 14  order_status      10000

In [12]:
df.isnull().sum()

order_id            0
order_date          0
delivery_date       0
business_id         0
business_name       0
business_type       0
business_address    0
contact_name        0
contact_phone       0
flour_type          0
quantity_bags       0
price_per_bag       0
total_amount        0
payment_method      0
order_status        0
rider_name          0
rider_phone         0
dtype: int64

In [13]:
# data transformation
dim_business = df[['business_id','business_name','business_type','business_address', 'contact_name', 'contact_phone']].copy().drop_duplicates(subset ='business_id').reset_index(drop =True)
dim_business.head()

,business_id,business_name,business_type,business_address,contact_name,contact_phone
0,BIZ-1018,"Adams, Zuniga and Wong",Restaurant,"Herbert Macaulay Way, Abuja",Elimu Agbaje,8017507864
1,BIZ-1006,Blake and Sons,Bakery,"Ahmadu Bello Way, Abuja",Bolanle Kalumba,8055667651
2,BIZ-1052,Chapman and Sons,Cafe,"Ahmadu Bello Way, Abuja",Hassan Nyoni,8083863413
3,BIZ-1035,Rodriguez-Graham,Restaurant,"Herbert Macaulay Way, Abuja",Mandela Onyango,8075228535
4,BIZ-1039,"Romero, Gonzalez and Brooks",Unknown,"Garki Area 1, Abuja",Mojisola Seko,8060119651


In [14]:
dim_rider = df[['rider_name', 'rider_phone']].copy().drop_duplicates().reset_index(drop = True)
dim_rider['rider_id']= dim_rider.index +1
dim_rider = dim_rider[['rider_id','rider_name', 'rider_phone']]
dim_rider.head()

,rider_id,rider_name,rider_phone
0,1,Aisha Bello,8089864260
1,2,Tunde Oladipo,8019121552
2,3,Emeka John,8019196777
3,4,Grace Onyema,8041568532


In [15]:
df.head()

,order_id,order_date,delivery_date,business_id,business_name,business_type,business_address,contact_name,contact_phone,flour_type,quantity_bags,price_per_bag,total_amount,payment_method,order_status,rider_name,rider_phone
0,ORD-214576,2025-10-25,2025-10-25,BIZ-1018,"Adams, Zuniga and Wong",Restaurant,"Herbert Macaulay Way, Abuja",Elimu Agbaje,8017507864,Unknown,26,9500.0,247000.0,POS,Delivered,Aisha Bello,8089864260
1,ORD-299448,2025-10-08,2025-10-08,BIZ-1006,Blake and Sons,Bakery,"Ahmadu Bello Way, Abuja",Bolanle Kalumba,8055667651,Bread Flour,27,10000.0,270000.0,POS,Cancelled,Tunde Oladipo,8019121552
2,ORD-246991,2025-10-17,2025-10-17,BIZ-1052,Chapman and Sons,Cafe,"Ahmadu Bello Way, Abuja",Hassan Nyoni,8083863413,Pastry Flour,21,9800.0,205800.0,Bank Transfer,Cancelled,Tunde Oladipo,8019121552
3,ORD-392075,2025-10-13,2025-10-13,BIZ-1035,Rodriguez-Graham,Restaurant,"Herbert Macaulay Way, Abuja",Mandela Onyango,8075228535,All-purpose,20,10500.0,210000.0,Bank Transfer,Pending,Tunde Oladipo,8019121552
4,ORD-179046,2025-10-14,2025-10-14,BIZ-1039,"Romero, Gonzalez and Brooks",Unknown,"Garki Area 1, Abuja",Mojisola Seko,8060119651,Bread Flour,40,10000.0,380000.0,POS,Pending,Emeka John,8019196777


In [16]:
dim_flour_type = df[['flour_type']].copy().drop_duplicates().reset_index(drop = True)
dim_flour_type['flour_id'] = range(1, len(dim_flour_type)+1 )
dim_flour_type = dim_flour_type[['flour_id','flour_type']]
dim_flour_type.head()

,flour_id,flour_type
0,1,Unknown
1,2,Bread Flour
2,3,Pastry Flour
3,4,All-purpose
4,5,Whole Wheat


In [17]:
orders_fact = df.copy()

orders_fact = orders_fact.merge(dim_rider, on =['rider_name', 'rider_phone'], how ='left')\
                       .merge(dim_flour_type, on = 'flour_type', how ='left')\
                        [['order_id','business_id','rider_id', 'flour_id','order_date','total_amount','quantity_bags', 'price_per_bag','payment_method','order_status']]
orders_fact.head()

,order_id,business_id,rider_id,flour_id,order_date,total_amount,quantity_bags,price_per_bag,payment_method,order_status
0,ORD-214576,BIZ-1018,1,1,2025-10-25,247000.0,26,9500.0,POS,Delivered
1,ORD-299448,BIZ-1006,2,2,2025-10-08,270000.0,27,10000.0,POS,Cancelled
2,ORD-246991,BIZ-1052,2,3,2025-10-17,205800.0,21,9800.0,Bank Transfer,Cancelled
3,ORD-392075,BIZ-1035,2,4,2025-10-13,210000.0,20,10500.0,Bank Transfer,Pending
4,ORD-179046,BIZ-1039,3,2,2025-10-14,380000.0,40,10000.0,POS,Pending


In [25]:
#export to csv
dim_rider.to_csv(r'C:\Users\Admin\Desktop\10Alytics\flour4four_etl_with_airflow\cleaned_dataset\dim_rider.csv', index = False)
dim_business.to_csv(r'C:\Users\Admin\Desktop\10Alytics\flour4four_etl_with_airflow\cleaned_dataset\dim_business.csv', index = False)
dim_flour_type.to_csv(r'C:\Users\Admin\Desktop\10Alytics\flour4four_etl_with_airflow\cleaned_dataset\dim_flour_type.csv', index = False)
orders_fact.to_csv(r'C:\Users\Admin\Desktop\10Alytics\flour4four_etl_with_airflow\cleaned_dataset\orders_fact.csv', index = False)

In [27]:
dim_business.head()

,business_id,business_name,business_type,business_address,contact_name,contact_phone
0,BIZ-1018,"Adams, Zuniga and Wong",Restaurant,"Herbert Macaulay Way, Abuja",Elimu Agbaje,8017507864
1,BIZ-1006,Blake and Sons,Bakery,"Ahmadu Bello Way, Abuja",Bolanle Kalumba,8055667651
2,BIZ-1052,Chapman and Sons,Cafe,"Ahmadu Bello Way, Abuja",Hassan Nyoni,8083863413
3,BIZ-1035,Rodriguez-Graham,Restaurant,"Herbert Macaulay Way, Abuja",Mandela Onyango,8075228535
4,BIZ-1039,"Romero, Gonzalez and Brooks",Unknown,"Garki Area 1, Abuja",Mojisola Seko,8060119651


In [19]:
#data loading
db_name = 'flour4four'
user = 'postgres'
password = 'postgres'
host = 'localhost'
port = '5432'

In [20]:
def get_connection():
    connection = psycopg2.connect(dbname = db_name,
                                  user = user,
                                  password = password,
                                  host = host,
                                  port = port)
    return connection


In [21]:
def create_tables():
    conn = get_connection()
    cursor = conn.cursor()
    table_query = '''CREATE SCHEMA IF NOT EXISTS flour;

                    DROP TABLE IF EXISTS flour.dim_business CASCADE;
                    DROP TABLE IF  EXISTS flour.dim_rider CASCADE;
                    DROP TABLE IF  EXISTS flour.dim_flour_type CASCADE;
                    DROP TABLE IF  EXISTS flour.orders_fact CASCADE;


                    CREATE TABLE IF NOT EXISTS flour.dim_business (
                    business_id VARCHAR  PRIMARY KEY,
                    business_name VARCHAR NOT NULL,
                    business_type VARCHAR ,
                    business_address VARCHAR,
                    contact_name VARCHAR NOT NULL, 
                    contact_phone VARCHAR NOT NULL
                    );



                    CREATE TABLE IF  NOT EXISTS flour.dim_rider (
                    rider_id  INT PRIMARY KEY,
                    rider_name VARCHAR NOT NULL, 
                    rider_phone VARCHAR NOT NULL);
                    

                    CREATE TABLE IF NOT EXISTS flour.dim_flour_type (
                    flour_id INT PRIMARY KEY,
                    flour_type VARCHAR
                    );

                    CREATE TABLE IF NOT EXISTS flour.orders_fact (
                    order_id VARCHAR PRIMARY KEY,
                    business_id VARCHAR NOT NULL REFERENCES flour.dim_business (business_id),
                    rider_id INT NOT NULL REFERENCES flour.dim_rider (rider_id), 
                    flour_id INT NOT NULL  REFERENCES flour.dim_flour_type (flour_id),
                    order_date  DATE NOT NULL,
                    total_amount FLOAT NOT NULL,
                    quantity_bags INT NOT NULL, 
                    price_per_bag FLOAT NOT NULL,
                    payment_method VARCHAR NOT NULL,
                    order_status VARCHAR NOT NULL
                    
                    );

'''
    cursor.execute(table_query)
    conn.commit()
    cursor.close()
    conn.close()
    print('Tables created sucessfully')

In [22]:
create_tables()

Tables created sucessfully


In [23]:

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db_name}")

In [24]:
dim_business.to_sql('dim_business', engine, schema = "flour", if_exists = 'append', index = False)
dim_rider.to_sql('dim_rider', engine, schema = "flour", if_exists = 'append', index = False)
dim_flour_type.to_sql('dim_flour_type', engine, schema = "flour", if_exists = 'append', index = False)
orders_fact.to_sql('orders_fact', engine, schema = "flour", if_exists = 'append', index = False)
print ('Data loaded to db sucessfully')

Data loaded to db sucessfully
